In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the path to the zip file
zip_file_path = '/content/archive (1).zip'

# Define the directory to extract the contents
extract_dir = '/content/extracted_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# List the contents of the extracted directory to find the data file
print(f"Contents of '{extract_dir}':")
for item in os.listdir(extract_dir):
    print(item)

# Assuming there's a CSV file, let's try to load it. Adjust filename if needed.
# You might need to change 'your_data_file.csv' to the actual CSV file name found above.
# For example, if it's 'Admission_Predict_Ver1.1.csv', use that.
data_file_name = None
for file in os.listdir(extract_dir):
    if file.endswith('.csv'):
        data_file_name = file
        break

if data_file_name:
    csv_file_path = os.path.join(extract_dir, data_file_name)
    df = pd.read_csv(csv_file_path)
    print(f"\nSuccessfully loaded '{data_file_name}' into a DataFrame.")
    display(df.head())
else:
    print("\nNo CSV file found in the extracted directory. Please check the contents manually.")
    df = None # Set df to None if no file is found

In [ ]:
if df is not None:
    print("DataFrame Info:")
    df.info()

    print("\nDataFrame Description:")
    display(df.describe().T)

    print("\nMissing values per column:")
    display(df.isnull().sum().sort_values(ascending=False))

    print("\nNumber of duplicate rows:", df.duplicated().sum())

In [ ]:
if df is not None:
    # Example: Fill numerical missing values with median, categorical with mode
    # For this dataset, usually no missing values are present. If there were:
    for column in df.columns:
        if df[column].isnull().any():
            if df[column].dtype in ['int64', 'float64']:
                df[column] = df[column].fillna(df[column].median())
                print(f"Filled missing values in '{column}' with median.")
            else:
                df[column] = df[column].fillna(df[column].mode()[0])
                print(f"Filled missing values in '{column}' with mode.")
    print("\nMissing values after handling:")
    display(df.isnull().sum())

In [ ]:
if df is not None:
    initial_rows = df.shape[0]
    df.drop_duplicates(inplace=True)
    rows_after_dropping = df.shape[0]
    print(f"Dropped {initial_rows - rows_after_dropping} duplicate rows.")
    print(f"DataFrame shape after dropping duplicates: {df.shape}")

In [ ]:
if df is not None:
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

    # Drop 'Serial No.' if it exists and is just an identifier
    if 'Serial No.' in numerical_cols:
        numerical_cols.remove('Serial No.')

    if numerical_cols:
        plt.figure(figsize=(15, 10))
        for i, col in enumerate(numerical_cols):
            plt.subplot(int(np.ceil(len(numerical_cols)/3)), 3, i + 1)
            sns.boxplot(y=df[col])
            plt.title(f'Box Plot of {col}')
            plt.ylabel('')
        plt.tight_layout()
        plt.show()
    else:
        print("No numerical columns for outlier identification.")

    # Example of IQR-based outlier detection (not removing, just identifying)
    # for col in numerical_cols:
    #     Q1 = df[col].quantile(0.25)
    #     Q3 = df[col].quantile(0.75)
    #     IQR = Q3 - Q1
    #     lower_bound = Q1 - 1.5 * IQR
    #     upper_bound = Q3 + 1.5 * IQR
    #     outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    #     if not outliers.empty:
    #         print(f"\nOutliers detected in '{col}': {len(outliers)} rows")
    #         # display(outliers[['Serial No.', col]].head()) # Adjust columns as needed

In [ ]:
if df is not None:
    # Histograms for numerical features
    plt.figure(figsize=(18, 12))
    for i, col in enumerate(numerical_cols):
        plt.subplot(int(np.ceil(len(numerical_cols)/3)), 3, i + 1)
        sns.histplot(df[col], kde=True)
        plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.show()

    # Count plots for categorical features (if any)
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_cols:
        plt.figure(figsize=(15, 5 * int(np.ceil(len(categorical_cols)/2))))
        for i, col in enumerate(categorical_cols):
            plt.subplot(int(np.ceil(len(categorical_cols)/2)), 2, i + 1)
            sns.countplot(y=df[col], order=df[col].value_counts().index)
            plt.title(f'Count of {col}')
        plt.tight_layout()
        plt.show()
    else:
        print("No categorical columns for count plots.")

In [ ]:
if df is not None:
    correlation_matrix = df[numerical_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Matrix of Numerical Features')
    plt.show()

In [ ]:
if df is not None:
    target_variable = 'Chance of Admit ' # Adjust this if your target column name is different

    if target_variable in df.columns:
        plt.figure(figsize=(18, 12))
        features_to_plot = [col for col in numerical_cols if col != target_variable]
        for i, col in enumerate(features_to_plot):
            plt.subplot(int(np.ceil(len(features_to_plot)/3)), 3, i + 1)
            sns.scatterplot(x=df[col], y=df[target_variable])
            plt.title(f'{col} vs. {target_variable}')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Target variable '{target_variable}' not found in DataFrame. Skipping scatter plots against target.")

In [ ]:
if df is not None and target_variable in df.columns:
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_cols:
        plt.figure(figsize=(15, 5 * int(np.ceil(len(categorical_cols)/2))))
        for i, col in enumerate(categorical_cols):
            plt.subplot(int(np.ceil(len(categorical_cols)/2)), 2, i + 1)
            sns.boxplot(x=df[col], y=df[target_variable])
            plt.title(f'{target_variable} by {col}')
            plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("No categorical columns to plot against target variable.")